# 0 - Setup

In [ ]:

!pip install pytorch-tabnet
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pytorch_tabnet.tab_model import TabNetClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.2 MB/s eta 0:00:00


### Import Dataset

In [ ]:
df = pd.read_csv("diabetes_data_upload.csv")
df.head()

,Age,Gender,Polyuria,Polydipsia,sudden weight loss,weakness,Polyphagia,Genital thrush,visual blurring,Itching,Irritability,delayed healing,partial paresis,muscle stiffness,Alopecia,Obesity,class
0,40,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes,Positive
1,58,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No,Positive
2,41,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No,Positive
3,45,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No,Positive
4,60,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Positive


# 1 - Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)
print("\nSummary stats:")
print(df.describe())

Shape: (520, 17)

Columns: Index(['Age', 'Gender', 'Polyuria', 'Polydipsia', 'sudden weight loss',
       'weakness', 'Polyphagia', 'Genital thrush', 'visual blurring',
       'Itching', 'Irritability', 'delayed healing', 'partial paresis',
       'muscle stiffness', 'Alopecia', 'Obesity', 'class'],
      dtype='object')

Summary stats:
              Age
count  520.000000
mean    48.028846
std     12.151466
min     16.000000
25%     39.000000
50%     47.500000
75%     57.000000
max     90.000000


# 2 - Encoding Categorical Columns

This dataset contains categorical features which must be converted into numerical values before training.
Tabnet, Like most neural network models, requires numerical input
Therefore, categorical variables are encoded into binary values to make them suitable for model training.


In [ ]:
df.replace({
    "Yes" : 1,
    "No" : 0,
    "Positive" : 1,
    "Negative" : 0,
    "Male" : 1,
    "Female" : 0

}, inplace=True)

/tmp/ipykernel_56706/2063388841.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({


In [ ]:
print(df.describe())

              Age      Gender    Polyuria  Polydipsia  sudden weight loss  \
count  520.000000  520.000000  520.000000  520.000000          520.000000   
mean    48.028846    0.630769    0.496154    0.448077            0.417308   
std     12.151466    0.483061    0.500467    0.497776            0.493589   
min     16.000000    0.000000    0.000000    0.000000            0.000000   
25%     39.000000    0.000000    0.000000    0.000000            0.000000   
50%     47.500000    1.000000    0.000000    0.000000            0.000000   
75%     57.000000    1.000000    1.000000    1.000000            1.000000   
max     90.000000    1.000000    1.000000    1.000000            1.000000   

         weakness  Polyphagia  Genital thrush  visual blurring     Itching  \
count  520.000000  520.000000      520.000000       520.000000  520.000000   
mean     0.586538    0.455769        0.223077         0.448077    0.486538   
std      0.492928    0.498519        0.416710         0.497776    0.5003

# 3 - TabNet Model

In [ ]:
X = df.drop('class', axis=1).to_numpy()  #features all columns expect target numpy since tabnet expects arrays
y = df['class'].to_numpy()     #target variable numpy since tabnet expects arrays


# first split: train+val vs test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# second split: train vs validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1
)

#fit the model
tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=128
)



/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.2424  | val_auc: 0.64964 |  0:00:00s
epoch 1  | loss: 0.89592 | val_auc: 0.53005 |  0:00:00s
epoch 2  | loss: 0.69057 | val_auc: 0.55889 |  0:00:00s
epoch 3  | loss: 0.54974 | val_auc: 0.58233 |  0:00:00s
epoch 4  | loss: 0.44869 | val_auc: 0.65144 |  0:00:01s
epoch 5  | loss: 0.42256 | val_auc: 0.70343 |  0:00:01s
epoch 6  | loss: 0.43321 | val_auc: 0.74519 |  0:00:01s
epoch 7  | loss: 0.33761 | val_auc: 0.81671 |  0:00:01s
epoch 8  | loss: 0.32538 | val_auc: 0.8152  |  0:00:01s
epoch 9  | loss: 0.26177 | val_auc: 0.80619 |  0:00:01s
epoch 10 | loss: 0.25811 | val_auc: 0.80439 |  0:00:01s
epoch 11 | loss: 0.22712 | val_auc: 0.78786 |  0:00:01s
epoch 12 | loss: 0.26152 | val_auc: 0.77885 |  0:00:02s
epoch 13 | loss: 0.23015 | val_auc: 0.8113  |  0:00:02s
epoch 14 | loss: 0.16061 | val_auc: 0.78936 |  0:00:02s
epoch 15 | loss: 0.15646 | val_auc: 0.76412 |  0:00:02s
epoch 16 | loss: 0.14723 | val_auc: 0.7491  |  0:00:02s
epoch 17 | loss: 0.12083 | val_auc: 0.71845 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  To improve performance additional hyperparameter tuning may be explored later. Default tabnet parameters were used above.

In [ ]:
#predict
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]

In [ ]:
print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

TabNet Results
Accuracy: 0.7019230769230769
Precision: 0.7142857142857143
Recall: 0.859375
F1 Score: 0.7801418439716312
ROC-AUC: 0.8703124999999999


The TabNet model demonstrated strong learning behavior, with validation AUC improving across early epochs and peaking at 0.82 before early stopping was triggered. This indicates that the model was able to effectively learn patterns from the data while avoiding overfitting. Final evaluation results show strong recall and ROC-AUC, suggesting the model performs well in identifying diabetic cases.

Multiple evaluation metrics were used to assess model performance, including Accuracy, Precision, Recall, F1 Score, and ROC-AUC. While accuracy provides an overall measure, recall is emphasized due to the importance of correctly identifying diabetic cases.


---


Accuracy - Model correct predicts about 72% of cases


---


Precisions - when model predicts diabetes, it is correct about 73% of the time which indicates a relatively high number of false positives


---


Recall - Model correctly identifies abbout 85% of actual diabetic cases which means it misses about 15%


---


F1 Score - Shows provides a balance of precision and recall , this indeicates how well the model is able to both correctly identify diabetic cases and minimize incorrect predictions This f1 score is 0.79 indicating strong and well-balanced performance. this suggests that the model is effective at both identifying diabetic cases and minimizing incorrect predictions.


---


Roc-Auc- The ROC-AUC score of 0.87 indicates strong model performance, demonstrating that the model is highly effective at distinguishing between diabetic and non-diabetic cases.

# 4 - Parameter Tuning for TabNet ( n_d, n_a )

The baseline TabNet model was trained using default parameters on the Early Diabetes Risk dataset,
achieving an accuracy of approximately 70% and a strong recall of 0.85 and ROC-AUC of 0.87.
The dataset contains 520 samples and 17 features, all of which were encoded from categorical
Yes/No and gender values into binary numerical format before training.

Since this is a medical prediction task, recall is the most important metric, as missing diabetic
patients (false negatives) is more critical than false positives. The baseline already demonstrates
strong recall, so tuning will explore whether performance can be further improved or stabilized.

To ensure fair comparison across experiments, the same training, validation, and test splits were
kept fixed while only TabNet hyperparameters and training settings were adjusted.

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1,
    n_d=16,    #according to paper n_d = n_a is a reasonable choice for most datasets DEFAULT
    n_a=16,
)

#fit the model
tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=128
)
#predict
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]

print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.54164 | val_auc: 0.57091 |  0:00:00s
epoch 1  | loss: 0.81215 | val_auc: 0.37981 |  0:00:00s
epoch 2  | loss: 0.44388 | val_auc: 0.51713 |  0:00:00s
epoch 3  | loss: 0.39136 | val_auc: 0.60847 |  0:00:00s
epoch 4  | loss: 0.37591 | val_auc: 0.56581 |  0:00:00s
epoch 5  | loss: 0.32327 | val_auc: 0.61058 |  0:00:00s
epoch 6  | loss: 0.29678 | val_auc: 0.60487 |  0:00:01s
epoch 7  | loss: 0.26427 | val_auc: 0.59645 |  0:00:01s
epoch 8  | loss: 0.21499 | val_auc: 0.60156 |  0:00:01s
epoch 9  | loss: 0.16103 | val_auc: 0.63221 |  0:00:01s
epoch 10 | loss: 0.17447 | val_auc: 0.64123 |  0:00:01s
epoch 11 | loss: 0.13548 | val_auc: 0.62831 |  0:00:01s
epoch 12 | loss: 0.13341 | val_auc: 0.61869 |  0:00:01s
epoch 13 | loss: 0.15231 | val_auc: 0.6265  |  0:00:01s
epoch 14 | loss: 0.13563 | val_auc: 0.63852 |  0:00:02s
epoch 15 | loss: 0.15896 | val_auc: 0.60006 |  0:00:02s
epoch 16 | loss: 0.12063 | val_auc: 0.628   |  0:00:02s
epoch 17 | loss: 0.1284  | val_auc: 0.63702 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Increasing the hidden dimension to n_d = n_a = 16 produced a dramatic improvement over the baseline, with recall reaching 1.0 — meaning the model correctly identified every diabetic case in the test set — and ROC-AUC jumping from 0.87 to 0.96. Accuracy improved to 0.875 and F1 score reached 0.91, indicating strong overall performance. The model trained steadily across 40 epochs, peaking at epoch 30 with a best validation AUC of 0.949 before early stopping triggered. Unlike the Pima Indians dataset where increasing n_d had no benefit, the Early Diabetes Risk dataset has 17 features compared to 8, giving the larger embedding width more meaningful structure to learn from. This suggests that n_d = n_a = 16 is a well-suited configuration for this dataset and represents a significant improvement over the default parameters.

# 5 - Tuning Parameters (n_steps & gamma)

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1,
    n_steps=5,  #according to paper n_steps [3,10] is optimal. With 17 features 5 is a reasonable midpoint
    gamma=1.5,  #according to paper larger n_steps favors a larger gamma
)

#fit the model
tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=128
)
#predict
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]
print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.33755 | val_auc: 0.5643  |  0:00:00s
epoch 1  | loss: 0.9467  | val_auc: 0.46995 |  0:00:00s
epoch 2  | loss: 0.76792 | val_auc: 0.45733 |  0:00:00s
epoch 3  | loss: 0.71455 | val_auc: 0.60637 |  0:00:00s
epoch 4  | loss: 0.65552 | val_auc: 0.57452 |  0:00:00s
epoch 5  | loss: 0.67722 | val_auc: 0.50481 |  0:00:00s
epoch 6  | loss: 0.69635 | val_auc: 0.49159 |  0:00:00s
epoch 7  | loss: 0.62616 | val_auc: 0.65445 |  0:00:00s
epoch 8  | loss: 0.53043 | val_auc: 0.71394 |  0:00:00s
epoch 9  | loss: 0.52768 | val_auc: 0.73137 |  0:00:00s
epoch 10 | loss: 0.45674 | val_auc: 0.77644 |  0:00:00s
epoch 11 | loss: 0.42052 | val_auc: 0.82933 |  0:00:00s
epoch 12 | loss: 0.37004 | val_auc: 0.84916 |  0:00:00s
epoch 13 | loss: 0.48587 | val_auc: 0.84195 |  0:00:00s
epoch 14 | loss: 0.46032 | val_auc: 0.82091 |  0:00:00s
epoch 15 | loss: 0.45243 | val_auc: 0.83353 |  0:00:00s
epoch 16 | loss: 0.46743 | val_auc: 0.85216 |  0:00:00s
epoch 17 | loss: 0.43504 | val_auc: 0.87139 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Setting n_steps=5 and gamma=1.5 produced strong but slightly lower performance compared to the n_d=16 experiment, with recall dropping from 1.0 to 0.953 and accuracy from 0.875 to 0.817. ROC-AUC remained high at 0.951, indicating the model still distinguishes well between classes overall. The model trained for 48 epochs before early stopping, peaking at epoch 38 with a best validation AUC of 0.935. The training curve shows more instability compared to the n_d=16 run, with validation AUC fluctuating noticeably across epochs rather than climbing steadily. On a dataset of only 520 samples, 5 attention steps introduces more complexity than necessary — the model has to distribute attention across more decision steps with fewer training examples to guide each one, leading to noisier convergence. While this configuration is competitive, the n_d=16 result remains superior for this dataset.

# 6 - Tune Performance (batch_size)

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1,
)

#fit the model
tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=64,           #according to paper batch size should be 1-10% of dataset size. Early Diabetes is small so we use a smaller batch
    virtual_batch_size=32    #virtual batch size kept proportionally smaller than batch_size
)

#predict
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]

print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

epoch 0  | loss: 0.8854  | val_auc: 0.70703 |  0:00:00s


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 1  | loss: 0.59945 | val_auc: 0.34585 |  0:00:00s
epoch 2  | loss: 0.47963 | val_auc: 0.79026 |  0:00:00s
epoch 3  | loss: 0.4767  | val_auc: 0.79177 |  0:00:00s
epoch 4  | loss: 0.38709 | val_auc: 0.88011 |  0:00:00s
epoch 5  | loss: 0.37083 | val_auc: 0.8774  |  0:00:00s
epoch 6  | loss: 0.36553 | val_auc: 0.8738  |  0:00:00s
epoch 7  | loss: 0.29599 | val_auc: 0.86659 |  0:00:00s
epoch 8  | loss: 0.32194 | val_auc: 0.89363 |  0:00:01s
epoch 9  | loss: 0.24268 | val_auc: 0.86629 |  0:00:01s
epoch 10 | loss: 0.23992 | val_auc: 0.8747  |  0:00:01s
epoch 11 | loss: 0.21491 | val_auc: 0.91526 |  0:00:01s
epoch 12 | loss: 0.23575 | val_auc: 0.93329 |  0:00:01s
epoch 13 | loss: 0.22699 | val_auc: 0.92548 |  0:00:01s
epoch 14 | loss: 0.20887 | val_auc: 0.91767 |  0:00:01s
epoch 15 | loss: 0.18806 | val_auc: 0.90685 |  0:00:01s
epoch 16 | loss: 0.15323 | val_auc: 0.90625 |  0:00:01s
epoch 17 | loss: 0.13667 | val_auc: 0.91526 |  0:00:02s
epoch 18 | loss: 0.12934 | val_auc: 0.94651 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Reducing the batch size to 64 produced the strongest results of all experiments, achieving 0.98 across accuracy, precision, recall, and F1 score, with a ROC-AUC of 0.999 — nearly perfect classification. The model trained steadily across 43 epochs, peaking at epoch 33 with a best validation AUC of 0.991. Unlike the Pima Indians dataset where reducing batch size caused instability and erratic predictions, the Early Diabetes Risk dataset benefited significantly from smaller batches. With 17 categorical features encoded into binary values, the data has a very structured and consistent pattern — smaller batches allow the model to update weights more frequently and pick up on these fine-grained feature combinations more effectively. The near-perfect balance between precision and recall at 0.984 indicates the model is not biased toward either class, making this the most clinically reliable configuration tested so far.

# 7 - Tune Performance (learning rate)

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1,
    optimizer_params=dict(lr=0.01)
)

#fit the model
tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=128
)

#predict
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]

print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

epoch 0  | loss: 1.2424  | val_auc: 0.49579 |  0:00:00s


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 1  | loss: 1.03343 | val_auc: 0.59195 |  0:00:00s
epoch 2  | loss: 0.87817 | val_auc: 0.52043 |  0:00:00s
epoch 3  | loss: 0.66809 | val_auc: 0.55108 |  0:00:00s
epoch 4  | loss: 0.63165 | val_auc: 0.61959 |  0:00:00s
epoch 5  | loss: 0.53964 | val_auc: 0.65204 |  0:00:00s
epoch 6  | loss: 0.48225 | val_auc: 0.71575 |  0:00:00s
epoch 7  | loss: 0.47761 | val_auc: 0.72596 |  0:00:00s
epoch 8  | loss: 0.46837 | val_auc: 0.76022 |  0:00:00s
epoch 9  | loss: 0.43772 | val_auc: 0.79808 |  0:00:00s
epoch 10 | loss: 0.36245 | val_auc: 0.81731 |  0:00:00s
epoch 11 | loss: 0.37062 | val_auc: 0.76743 |  0:00:00s
epoch 12 | loss: 0.36657 | val_auc: 0.72356 |  0:00:00s
epoch 13 | loss: 0.3177  | val_auc: 0.71605 |  0:00:00s
epoch 14 | loss: 0.28209 | val_auc: 0.73918 |  0:00:00s
epoch 15 | loss: 0.28311 | val_auc: 0.76863 |  0:00:00s
epoch 16 | loss: 0.25252 | val_auc: 0.78275 |  0:00:00s
epoch 17 | loss: 0.24221 | val_auc: 0.79657 |  0:00:00s
epoch 18 | loss: 0.25611 | val_auc: 0.80108 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Reducing the learning rate to 0.01 produced competitive but noticeably lower performance compared to the batch size experiment, with accuracy and recall at 0.875 and 0.906 respectively and ROC-AUC dropping to 0.927. The model required significantly more epochs to converge, running until epoch 57 before early stopping triggered with a best validation AUC of 0.917 at epoch 47. The training curve shows a slow, steady climb rather than the sharp improvements seen in the batch size experiment, which is characteristic of a lower learning rate — the model takes smaller steps and converges more gradually. While the results are still strong and recall remains above 0.90, the slower convergence and lower ceiling compared to the batch size configuration suggests that 0.01 is too conservative a learning rate for this dataset. The default learning rate of 0.02 strikes a better balance between convergence speed and final performance.

# 8 - COMBO

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1,
    n_d=32,    #according to paper n_d = n_a is a reasonable choice for most datasets
    n_a=32,
    n_steps=5,  #according to paper n_steps [3,10] is optimal
    gamma=1.5,  #according to paper larger n_steps favors a larger gamma
)

#fit the model
tabnet_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=128
)
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]
print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


epoch 0  | loss: 2.58943 | val_auc: 0.45312 |  0:00:00s


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 1  | loss: 0.98725 | val_auc: 0.36358 |  0:00:00s
epoch 2  | loss: 0.71778 | val_auc: 0.39002 |  0:00:00s
epoch 3  | loss: 0.60988 | val_auc: 0.28245 |  0:00:00s
epoch 4  | loss: 0.39951 | val_auc: 0.39844 |  0:00:00s
epoch 5  | loss: 0.40007 | val_auc: 0.58534 |  0:00:00s
epoch 6  | loss: 0.45065 | val_auc: 0.60787 |  0:00:00s
epoch 7  | loss: 0.37734 | val_auc: 0.61538 |  0:00:00s
epoch 8  | loss: 0.29744 | val_auc: 0.5595  |  0:00:00s
epoch 9  | loss: 0.35507 | val_auc: 0.55709 |  0:00:00s
epoch 10 | loss: 0.33875 | val_auc: 0.55258 |  0:00:01s
epoch 11 | loss: 0.3452  | val_auc: 0.55258 |  0:00:01s
epoch 12 | loss: 0.37472 | val_auc: 0.53846 |  0:00:01s
epoch 13 | loss: 0.34657 | val_auc: 0.52284 |  0:00:01s
epoch 14 | loss: 0.28228 | val_auc: 0.58624 |  0:00:01s
epoch 15 | loss: 0.2122  | val_auc: 0.51713 |  0:00:01s
epoch 16 | loss: 0.27069 | val_auc: 0.52224 |  0:00:01s
epoch 17 | loss: 0.22145 | val_auc: 0.50691 |  0:00:01s

Early stopping occurred at epoch 17 with best_e

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


The combined configuration of n_d=32, n_a=32, n_steps=5, and gamma=1.5 produced the worst results of all experiments on this dataset, with accuracy dropping to 0.73, recall falling to 0.69, and ROC-AUC collapsing to 0.72. Early stopping triggered at epoch 17 with the best performance occurring at epoch 7 and a best validation AUC of only 0.66, indicating the model failed to learn meaningful patterns. The training curve shows severe instability with validation AUC fluctuating between 0.38 and 0.66 without any consistent improvement. This result demonstrates that combining multiple aggressive parameter changes compounds their individual weaknesses rather than combining their strengths — the larger embedding width, additional attention steps, and higher gamma together create a model that is far too complex for a dataset of only 520 samples. This is a key finding: the best performing configuration for this dataset was actually the simplest change — reducing batch size alone — rather than stacking multiple parameter modifications together.



# 9 - Size of the training dataset

In [ ]:
X_train_50, _, y_train_50, _ = train_test_split(
    X_train, y_train, test_size=0.5, random_state=42, stratify=y_train
)

X_train_75, _, y_train_75, _ = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1
)

#fit the model
tabnet_model.fit(
    X_train_50, y_train_50,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=32,
    virtual_batch_size=16
)

y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]

print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.91478 | val_auc: 0.64213 |  0:00:00s
epoch 1  | loss: 0.66451 | val_auc: 0.60697 |  0:00:00s
epoch 2  | loss: 0.63342 | val_auc: 0.63762 |  0:00:00s
epoch 3  | loss: 0.46145 | val_auc: 0.64002 |  0:00:00s
epoch 4  | loss: 0.36514 | val_auc: 0.75421 |  0:00:00s
epoch 5  | loss: 0.43864 | val_auc: 0.72656 |  0:00:00s
epoch 6  | loss: 0.50483 | val_auc: 0.71935 |  0:00:00s
epoch 7  | loss: 0.32587 | val_auc: 0.83654 |  0:00:00s
epoch 8  | loss: 0.40596 | val_auc: 0.79026 |  0:00:00s
epoch 9  | loss: 0.41527 | val_auc: 0.8131  |  0:00:01s
epoch 10 | loss: 0.36929 | val_auc: 0.78486 |  0:00:01s
epoch 11 | loss: 0.37683 | val_auc: 0.79026 |  0:00:01s
epoch 12 | loss: 0.34662 | val_auc: 0.80048 |  0:00:01s
epoch 13 | loss: 0.31076 | val_auc: 0.86298 |  0:00:01s
epoch 14 | loss: 0.29602 | val_auc: 0.86178 |  0:00:01s
epoch 15 | loss: 0.30032 | val_auc: 0.8756  |  0:00:01s
epoch 16 | loss: 0.264   | val_auc: 0.90865 |  0:00:01s
epoch 17 | loss: 0.19908 | val_auc: 0.91526 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Training on 50% of the data with a corrected batch size of 32 produced strong recall of 0.953 but lower accuracy of 0.71 and precision of 0.69, indicating the model is over-predicting diabetic cases when trained on fewer examples. The model peaked at epoch 12 with a best validation AUC of 0.931 before early stopping triggered at epoch 22. While recall remained high, the drop in precision and accuracy compared to the full training set shows that reducing data availability pushes the model toward predicting the positive class more aggressively to compensate for the reduced signal. This is a meaningful finding — even at half the training size the model retains strong sensitivity, but at the cost of more false positives.

# 75% Training

In [ ]:
#define tabnet model
tabnet_model = TabNetClassifier(
    seed=42,
    verbose=1
)

#fit the model
tabnet_model.fit(
    X_train_75, y_train_75,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=10,
    batch_size=32,
    virtual_batch_size=16
)

#predict
y_pred = tabnet_model.predict(X_test)
y_prob = tabnet_model.predict_proba(X_test)[:, 1]

print("TabNet Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.85495 | val_auc: 0.71394 |  0:00:00s
epoch 1  | loss: 0.56946 | val_auc: 0.75421 |  0:00:00s
epoch 2  | loss: 0.51053 | val_auc: 0.7476  |  0:00:00s
epoch 3  | loss: 0.47653 | val_auc: 0.75841 |  0:00:00s
epoch 4  | loss: 0.39837 | val_auc: 0.78425 |  0:00:00s
epoch 5  | loss: 0.36182 | val_auc: 0.8113  |  0:00:00s
epoch 6  | loss: 0.34629 | val_auc: 0.83594 |  0:00:01s
epoch 7  | loss: 0.31286 | val_auc: 0.82993 |  0:00:01s
epoch 8  | loss: 0.39167 | val_auc: 0.84675 |  0:00:01s
epoch 9  | loss: 0.32216 | val_auc: 0.86298 |  0:00:01s
epoch 10 | loss: 0.31668 | val_auc: 0.86298 |  0:00:01s
epoch 11 | loss: 0.33007 | val_auc: 0.86959 |  0:00:01s
epoch 12 | loss: 0.26452 | val_auc: 0.89844 |  0:00:01s
epoch 13 | loss: 0.22396 | val_auc: 0.93029 |  0:00:02s
epoch 14 | loss: 0.22186 | val_auc: 0.92728 |  0:00:02s
epoch 15 | loss: 0.16489 | val_auc: 0.92548 |  0:00:02s
epoch 16 | loss: 0.24361 | val_auc: 0.91707 |  0:00:02s
epoch 17 | loss: 0.27191 | val_auc: 0.92007 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Training on 75% of the data produced strong and well-balanced results, with accuracy of 0.90, precision of 0.982, and ROC-AUC of 0.964. The model trained steadily across 33 epochs, peaking at epoch 23 with a best validation AUC of 0.945. Compared to the 50% experiment, precision improved dramatically from 0.69 to 0.98 while recall dropped slightly from 0.953 to 0.859, indicating a more balanced and reliable model with less over-prediction of the positive class. The additional training data gives the model enough examples to learn a more precise decision boundary rather than defaulting to aggressive positive predictions.

##Quick Analysis

Across all experiments the Early Diabetes Risk dataset showed a very different pattern from the Pima Indians dataset. Rather than instability and collapse, most configurations produced competitive results, with the standout finding being that reducing batch size to 64 yielded near-perfect performance at 0.98 across all metrics and a ROC-AUC of 0.999. This improvement is attributed to the structured binary nature of the dataset — 17 categorical features encoded as Yes/No values form highly consistent patterns that smaller batches allow the model to learn more precisely through more frequent weight updates. The training size experiments confirmed that performance scales meaningfully with data availability, with 75% training size producing strong balanced results and 50% causing the model to over-predict positive cases. The only configuration that failed significantly was the combined parameter experiment, reinforcing that stacking multiple aggressive changes on a small dataset is counterproductive. Overall TabNet performs well on this dataset when batch size is appropriately matched to dataset size.